# 1. ADNI non-imaging inventory and coverage

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


# 2. Connect Google Drive

connect this notebook to Google Drive and confirm that the `adni_non_imaging` project folder is accessible. also display its existing subfolders so that I can verify that the raw data are preserved and the output folders are ready for the new preprocessing workflow.

In [ ]:
from pathlib import Path
from google.colab import drive

# Connect Google Drive to this Colab session.
drive.mount("/content/drive")

# Define the main folder for the non-imaging preprocessing workflow.
PROJECT_DIR = Path("/content/drive/MyDrive/adni_mri/adni_non_imaging")

# Confirm that the expected project folder exists.
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"The expected project folder was not found:\n{PROJECT_DIR}"
    )

print(f"Project folder connected successfully:\n{PROJECT_DIR}\n")

# Display the folders currently present inside the project directory.
print("Current project contents:")
for path in sorted(PROJECT_DIR.iterdir()):
    item_type = "folder" if path.is_dir() else "file"
    print(f"- {path.name} ({item_type})")

# 3. Define and validate the project folders

define the paths used throughout the non-imaging preprocessing workflow. then confirm that the raw-data folder contains files and check that the generated-output folders are empty before beginning the new analysis.

The raw files will remain unchanged. All inventories, quality-control reports, intermediate tables, processed data and final manifests will be written to their corresponding output folders.

In [ ]:
from pathlib import Path

# Define the main non-imaging project folder.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging"
)

# Define the folders used throughout the preprocessing workflow.
RAW_DIR = PROJECT_DIR / "raw"
INVENTORY_DIR = PROJECT_DIR / "inventory"
QC_DIR = PROJECT_DIR / "qc"
INTERIM_DIR = PROJECT_DIR / "interim"
PROCESSED_DIR = PROJECT_DIR / "processed"
MANIFESTS_DIR = PROJECT_DIR / "manifests"

PROJECT_FOLDERS = {
    "raw": RAW_DIR,
    "inventory": INVENTORY_DIR,
    "qc": QC_DIR,
    "interim": INTERIM_DIR,
    "processed": PROCESSED_DIR,
    "manifests": MANIFESTS_DIR,
}

# Confirm that every expected folder exists.
missing_folders = [
    str(path)
    for path in PROJECT_FOLDERS.values()
    if not path.exists()
]

if missing_folders:
    raise FileNotFoundError(
        "The following expected project folders were not found:\n"
        + "\n".join(missing_folders)
    )

# Find all files stored anywhere inside the raw-data folder.
raw_files = sorted(
    path
    for path in RAW_DIR.rglob("*")
    if path.is_file()
)

if not raw_files:
    raise FileNotFoundError(
        f"No raw files were found inside:\n{RAW_DIR}"
    )

print("Project folders validated successfully.\n")

for folder_name, folder_path in PROJECT_FOLDERS.items():
    files_inside = [
        path
        for path in folder_path.rglob("*")
        if path.is_file()
    ]

    print(
        f"{folder_name:<10} | "
        f"{len(files_inside):>4} file(s) | "
        f"{folder_path}"
    )

print(f"\nTotal preserved raw files found: {len(raw_files)}")

# 4. Create a read-only inventory of the raw files

create an inventory of every file stored inside the raw-data folder without opening or modifying the datasets. The inventory will record each file's location, name, extension, size and modification time.

This will confirm exactly which ADNI files are available before I begin inspecting their contents. The resulting inventory will be saved in the `inventory` folder.

In [ ]:
import pandas as pd
from datetime import datetime

# Collect metadata for every file stored inside the raw-data folder.
inventory_records = []

for file_path in sorted(RAW_DIR.rglob("*")):
    if not file_path.is_file():
        continue

    file_stat = file_path.stat()

    inventory_records.append(
        {
            "relative_path": str(file_path.relative_to(RAW_DIR)),
            "parent_folder": str(file_path.parent.relative_to(RAW_DIR)),
            "filename": file_path.name,
            "file_stem": file_path.stem,
            "extension": "".join(file_path.suffixes).lower(),
            "size_bytes": file_stat.st_size,
            "size_mb": round(file_stat.st_size / (1024 ** 2), 3),
            "modified_datetime": datetime.fromtimestamp(
                file_stat.st_mtime
            ).isoformat(timespec="seconds"),
        }
    )

# Convert the collected metadata into a structured table.
raw_file_inventory = pd.DataFrame(inventory_records)

if raw_file_inventory.empty:
    raise RuntimeError(
        f"No files were found inside the raw-data folder:\n{RAW_DIR}"
    )

# Sort the inventory so files from the same folder remain together.
raw_file_inventory = raw_file_inventory.sort_values(
    by=["parent_folder", "filename"],
    kind="stable",
).reset_index(drop=True)

# Save the inventory without modifying any raw datasets.
inventory_output_path = INVENTORY_DIR / "raw_file_inventory.csv"
raw_file_inventory.to_csv(inventory_output_path, index=False)

print(f"Raw-file inventory created successfully.")
print(f"Files recorded: {len(raw_file_inventory):,}")
print(f"Total raw-data size: {raw_file_inventory['size_mb'].sum():,.2f} MB")
print(f"Saved to:\n{inventory_output_path}")

display(raw_file_inventory)

# 5. Load and inspect the DXSUM diagnosis table

DXSUM will be the authoritative source for baseline diagnosis, diagnosis dates, longitudinal diagnosis changes, conversion to AD and stability during follow-up.

At this stage, only inspect the table structure, dimensions, column names and a small sample. not filter participants or create outcome labels yet.

In [ ]:
import pandas as pd

# Define the exact path to the preserved raw DXSUM table.
DXSUM_PATH = (
    RAW_DIR
    / "Cohort, dates and source-of-truth tables"
    / "All_Subjects_DXSUM_11Jul2026.csv"
)

# Confirm that the expected source file exists before loading it.
if not DXSUM_PATH.exists():
    raise FileNotFoundError(
        f"The DXSUM file was not found at:\n{DXSUM_PATH}"
    )

# Load the raw DXSUM table without modifying the source file.
dxsum_raw = pd.read_csv(
    DXSUM_PATH,
    low_memory=False,
)

print("DXSUM loaded successfully.")
print(f"Source file:\n{DXSUM_PATH}\n")

print(f"Rows: {len(dxsum_raw):,}")
print(f"Columns: {dxsum_raw.shape[1]:,}")

print("\nColumn names:")
print(dxsum_raw.columns.tolist())

print("\nFirst five rows:")
display(dxsum_raw.head())

# 6. Validate the core DXSUM fields

create a working copy of DXSUM and validate the fields needed for clinical cohort construction. check participant identifiers, diagnosis dates, diagnosis codes, ADNI phases, visit codes, duplicated records and any rows already marked with quality-control errors.

not remove any participants or diagnosis records at this stage. The purpose of this step is to identify structural problems before defining baseline diagnoses and longitudinal outcomes.

In [ ]:
import pandas as pd

# Define the fields required for clinical cohort construction.
required_dxsum_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",
    "DIAGNOSIS",
    "HAS_QC_ERROR",
]

missing_columns = [
    column
    for column in required_dxsum_columns
    if column not in dxsum_raw.columns
]

if missing_columns:
    raise KeyError(
        "DXSUM is missing the following required columns:\n"
        + "\n".join(missing_columns)
    )

# Create a working copy so that the raw dataframe remains unchanged.
dxsum = dxsum_raw.copy()

# Standardize text fields without altering the original source file.
for column in ["PHASE", "PTID", "VISCODE", "VISCODE2"]:
    dxsum[column] = dxsum[column].astype("string").str.strip()

# Parse participant identifiers, diagnosis codes and examination dates.
# Invalid values are preserved as missing so they can be inspected.
dxsum["RID_CLEAN"] = pd.to_numeric(
    dxsum["RID"],
    errors="coerce",
).astype("Int64")

dxsum["DIAGNOSIS_CODE"] = pd.to_numeric(
    dxsum["DIAGNOSIS"],
    errors="coerce",
).astype("Int64")

dxsum["EXAMDATE_PARSED"] = pd.to_datetime(
    dxsum["EXAMDATE"],
    errors="coerce",
)

# Add readable labels for the three diagnosis codes used by DXSUM.
diagnosis_labels = {
    1: "CN",
    2: "MCI",
    3: "AD",
}

dxsum["DIAGNOSIS_LABEL"] = (
    dxsum["DIAGNOSIS_CODE"]
    .map(diagnosis_labels)
    .astype("string")
)

# Identify exact duplicate clinical records.
duplicate_key = [
    "RID_CLEAN",
    "EXAMDATE_PARSED",
    "VISCODE2",
    "DIAGNOSIS_CODE",
]

exact_duplicate_mask = dxsum.duplicated(
    subset=duplicate_key,
    keep=False,
)

# Identify dates on which the same participant has more than one diagnosis.
same_day_diagnosis_counts = (
    dxsum.dropna(
        subset=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "DIAGNOSIS_CODE",
        ]
    )
    .groupby(
        ["RID_CLEAN", "EXAMDATE_PARSED"]
    )["DIAGNOSIS_CODE"]
    .nunique()
)

conflicting_same_day_keys = same_day_diagnosis_counts[
    same_day_diagnosis_counts > 1
]

# Identify diagnosis values outside the expected CN, MCI and AD codes.
unexpected_diagnosis_rows = dxsum[
    dxsum["DIAGNOSIS_CODE"].notna()
    & ~dxsum["DIAGNOSIS_CODE"].isin([1, 2, 3])
].copy()

# Create a compact audit summary.
dxsum_audit = pd.DataFrame(
    {
        "check": [
            "Total DXSUM rows",
            "Unique non-missing RIDs",
            "Unique non-missing PTIDs",
            "Rows with missing RID",
            "Rows with missing PTID",
            "Rows with missing examination date",
            "Rows with missing diagnosis",
            "Rows with unexpected diagnosis codes",
            "Rows involved in exact duplicate records",
            "Participant-date combinations with conflicting diagnoses",
        ],
        "count": [
            len(dxsum),
            dxsum["RID_CLEAN"].nunique(dropna=True),
            dxsum["PTID"].nunique(dropna=True),
            int(dxsum["RID_CLEAN"].isna().sum()),
            int(dxsum["PTID"].isna().sum()),
            int(dxsum["EXAMDATE_PARSED"].isna().sum()),
            int(dxsum["DIAGNOSIS_CODE"].isna().sum()),
            len(unexpected_diagnosis_rows),
            int(exact_duplicate_mask.sum()),
            len(conflicting_same_day_keys),
        ],
    }
)

# Save the audit summary as a quality-control report.
dxsum_audit_path = QC_DIR / "dxsum_core_field_audit.csv"
dxsum_audit.to_csv(dxsum_audit_path, index=False)

print("DXSUM core-field audit completed.")
print(f"Audit report saved to:\n{dxsum_audit_path}\n")

display(dxsum_audit)

print("\nDiagnosis distribution:")
display(
    dxsum[
        ["DIAGNOSIS_CODE", "DIAGNOSIS_LABEL"]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
)

print("\nADNI phase distribution:")
display(
    dxsum["PHASE"]
    .value_counts(dropna=False)
    .rename_axis("PHASE")
    .reset_index(name="row_count")
)

print("\nMost frequent VISCODE2 values:")
display(
    dxsum["VISCODE2"]
    .value_counts(dropna=False)
    .head(25)
    .rename_axis("VISCODE2")
    .reset_index(name="row_count")
)

print("\nHAS_QC_ERROR distribution:")
display(
    dxsum["HAS_QC_ERROR"]
    .value_counts(dropna=False)
    .rename_axis("HAS_QC_ERROR")
    .reset_index(name="row_count")
)

# Display suspicious records only when they exist.
if exact_duplicate_mask.any():
    print("\nExample exact duplicate records:")
    display(
        dxsum.loc[
            exact_duplicate_mask,
            [
                "PHASE",
                "PTID",
                "RID_CLEAN",
                "VISCODE",
                "VISCODE2",
                "EXAMDATE_PARSED",
                "DIAGNOSIS_CODE",
                "DIAGNOSIS_LABEL",
            ],
        ]
        .sort_values(
            ["RID_CLEAN", "EXAMDATE_PARSED", "VISCODE2"]
        )
        .head(20)
    )

if len(conflicting_same_day_keys) > 0:
    conflicting_index = set(conflicting_same_day_keys.index)

    conflicting_rows = dxsum[
        dxsum.apply(
            lambda row: (
                row["RID_CLEAN"],
                row["EXAMDATE_PARSED"],
            )
            in conflicting_index,
            axis=1,
        )
    ]

    print("\nExample same-day conflicting diagnoses:")
    display(
        conflicting_rows[
            [
                "PHASE",
                "PTID",
                "RID_CLEAN",
                "VISCODE",
                "VISCODE2",
                "EXAMDATE_PARSED",
                "DIAGNOSIS_CODE",
                "DIAGNOSIS_LABEL",
                "HAS_QC_ERROR",
            ]
        ]
        .sort_values(
            ["RID_CLEAN", "EXAMDATE_PARSED", "DIAGNOSIS_CODE"]
        )
        .head(20)
    )

## 6.1. What the initial DXSUM audit shows

The DXSUM table contains **16,247 diagnosis records from 3,711 participants**. The number of rows is much larger than the number of participants because most people were assessed repeatedly across baseline and follow-up visits. At this stage, these counts describe the complete diagnosis history available in DXSUM.

The participant identifiers are in very good condition. Every row has both an RID and a PTID, and both identifiers correspond to the same total of 3,711 unique participants. There are also no exact duplicate diagnosis records. This means that there is no immediate evidence that participants are being lost through missing identifiers or that identical records have been accidentally repeated.

The diagnosis coding is also structurally consistent. All non-missing diagnoses use the expected ADNI codes:

- `1` for cognitively normal (CN),
- `2` for mild cognitive impairment (MCI),
- `3` for Alzheimer's disease (AD).

Across all visits, there are 6,564 CN records, 6,610 MCI records and 3,027 AD records. These are **visit-level counts rather than participant counts**, so they should not be interpreted as the number of unique people in each diagnostic group. A participant may contribute several MCI records and later contribute one or more AD records after conversion.

There are **46 rows without a diagnosis** and **111 rows without an examination date**. These records cannot directly define a dated diagnosis event. However, they should not yet be deleted automatically because some may still contain useful administrative information or may be explainable through other fields. They will be examined separately before deciding whether they can contribute to cohort construction.

The data cover all major ADNI phases:

- ADNI1,
- ADNIGO,
- ADNI2,
- ADNI3,
- ADNI4.

ADNI2 contributes the largest number of diagnosis records, followed by ADNI1, ADNI3 and ADNI4. This confirms that the table contains a broad longitudinal population rather than only the older ADNI phases. The phase information will later be retained because follow-up availability, assessment schedules and missingness patterns may differ across phases.

The visit-code distribution follows the expected longitudinal structure. Baseline (`bl`) and screening (`sc`) are the most common early visit codes, followed by month 6, month 12, month 24 and later follow-up visits. Some participants have diagnosis records extending beyond 10 years. This long follow-up is useful for determining stable MCI, late conversion, reversion and other diagnostic trajectories.

Most records are not marked as having a quality-control error. Only **one row has `HAS_QC_ERROR = 1`**. The many missing values in this column should not automatically be interpreted as errors because older ADNI records may not have used this flag. The single explicitly flagged row will therefore be inspected rather than removed without review.

The main inconsistency found in this audit is that **two participant-date combinations contain conflicting diagnoses on the same calendar day**:

- RID 1200 is recorded as CN at `m24` and MCI at `m36` on 20 September 2010.
- RID 4855 is recorded as MCI at an ADNI2 visit and CN at an ADNI3 visit on 30 October 2018.

These are not exact duplicate rows. They appear to involve different visit codes or transitions between study phases that were assigned the same examination date. They must be reviewed carefully because the diagnosis trajectory cannot treat both same-day diagnoses as independent chronological events. Until a clear resolution rule is established, these cases should remain flagged rather than being silently assigned one diagnosis.

# 7. Inspect exceptional DXSUM records

examine the records identified during the initial audit:

- rows without an examination date,
- rows without a diagnosis,
- rows explicitly marked with a quality-control error,
- participants with conflicting diagnoses recorded on the same date.

preserve these records exactly as they appear in DXSUM and save separate quality-control tables for review. For missing examination dates, also inspect administrative dates such as `USERDATE` and `USERDATE2`, but not use them to replace the clinical examination date automatically.

For the two participants with same-day diagnostic conflicts, display their complete diagnosis histories. This will provide the surrounding visits needed to understand whether the conflict is caused by an incorrect date, overlapping ADNI phases, or a genuine inconsistency in the diagnosis records.

In [ ]:
# Parse the administrative date fields for inspection only.
# These dates will not replace EXAMDATE automatically.
for column in ["USERDATE", "USERDATE2"]:
    parsed_column = f"{column}_PARSED"

    if column in dxsum.columns:
        dxsum[parsed_column] = pd.to_datetime(
            dxsum[column],
            errors="coerce",
        )

# Standardize the explicit DXSUM quality-control flag.
dxsum["HAS_QC_ERROR_CLEAN"] = pd.to_numeric(
    dxsum["HAS_QC_ERROR"],
    errors="coerce",
).astype("Int64")

# Collect rows with missing clinical examination dates.
missing_examdate_rows = dxsum[
    dxsum["EXAMDATE_PARSED"].isna()
].copy()

# Collect rows with missing diagnosis values.
missing_diagnosis_rows = dxsum[
    dxsum["DIAGNOSIS_CODE"].isna()
].copy()

# Collect rows explicitly marked as having a QC error.
explicit_qc_error_rows = dxsum[
    dxsum["HAS_QC_ERROR_CLEAN"].eq(1)
].copy()

# Identify participant-date combinations containing more than one diagnosis.
valid_dated_diagnoses = dxsum.dropna(
    subset=[
        "RID_CLEAN",
        "EXAMDATE_PARSED",
        "DIAGNOSIS_CODE",
    ]
).copy()

same_day_diagnosis_count = (
    valid_dated_diagnoses
    .groupby(
        ["RID_CLEAN", "EXAMDATE_PARSED"]
    )["DIAGNOSIS_CODE"]
    .transform("nunique")
)

same_day_conflict_rows = valid_dated_diagnoses[
    same_day_diagnosis_count > 1
].copy()

# Extract the complete diagnosis histories of participants involved
# in same-day diagnostic conflicts.
conflict_rids = (
    same_day_conflict_rows["RID_CLEAN"]
    .dropna()
    .unique()
    .tolist()
)

conflict_participant_timelines = dxsum[
    dxsum["RID_CLEAN"].isin(conflict_rids)
].copy()

conflict_participant_timelines = (
    conflict_participant_timelines
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Define the most useful fields for reviewing problematic records.
review_columns = [
    "PHASE",
    "PTID",
    "RID_CLEAN",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",
    "EXAMDATE_PARSED",
    "DIAGNOSIS",
    "DIAGNOSIS_CODE",
    "DIAGNOSIS_LABEL",
    "DXNORM",
    "DXMCI",
    "DXAD",
    "DXCONFID",
    "USERDATE",
    "USERDATE2",
    "HAS_QC_ERROR",
]

review_columns = [
    column
    for column in review_columns
    if column in dxsum.columns
]

# Save each exception category as a separate QC table.
qc_outputs = {
    "dxsum_rows_missing_examdate.csv": missing_examdate_rows,
    "dxsum_rows_missing_diagnosis.csv": missing_diagnosis_rows,
    "dxsum_explicit_qc_error_rows.csv": explicit_qc_error_rows,
    "dxsum_same_day_conflicting_diagnoses.csv": same_day_conflict_rows,
    "dxsum_conflict_participant_full_timelines.csv": conflict_participant_timelines,
}

for filename, dataframe in qc_outputs.items():
    output_path = QC_DIR / filename
    dataframe.to_csv(output_path, index=False)

# Summarize the exception categories.
exception_summary = pd.DataFrame(
    {
        "exception_category": [
            "Rows without EXAMDATE",
            "Rows without DIAGNOSIS",
            "Rows explicitly marked HAS_QC_ERROR = 1",
            "Rows involved in same-day diagnosis conflicts",
            "Participants involved in same-day diagnosis conflicts",
        ],
        "count": [
            len(missing_examdate_rows),
            len(missing_diagnosis_rows),
            len(explicit_qc_error_rows),
            len(same_day_conflict_rows),
            len(conflict_rids),
        ],
    }
)

exception_summary_path = (
    QC_DIR / "dxsum_exception_record_summary.csv"
)
exception_summary.to_csv(
    exception_summary_path,
    index=False,
)

print("DXSUM exception records inspected and saved.")
print(f"Summary saved to:\n{exception_summary_path}\n")

display(exception_summary)

print("\nRows without an examination date:")
display(
    missing_examdate_rows[review_columns].head(20)
)

print("\nRows without a diagnosis:")
display(
    missing_diagnosis_rows[review_columns].head(20)
)

print("\nRows explicitly marked with a QC error:")
display(
    explicit_qc_error_rows[review_columns]
)

print("\nSame-day conflicting diagnosis records:")
display(
    same_day_conflict_rows[review_columns].sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
        ]
    )
)

print("\nComplete diagnosis histories for the affected participants:")
display(
    conflict_participant_timelines[review_columns]
)

# 8. Determine whether exceptional rows affect participant eligibility

examine the participants associated with missing diagnosis dates or missing diagnosis values.

not drop these participants automatically. Instead, determine whether each person still has:

- at least one valid dated diagnosis,
- a valid formal baseline diagnosis recorded at `VISCODE2 = bl`,
- later dated diagnosis records after baseline,
- a usable first and final diagnosis in their remaining DXSUM history.

This distinction is important because an incomplete row does not necessarily make the entire participant unusable. If a participant has other complete diagnosis records, I can exclude only the incomplete row from the clinical timeline while retaining the participant.

Participants without a valid formal baseline diagnosis cannot enter the baseline-defined cohort. Participants with a valid baseline but insufficient subsequent diagnosis information may still contribute to baseline CN or AD analyses, but they may not be suitable for assigning pMCI or sMCI outcomes.

The resulting subject-level audit will be saved in the quality-control folder before any rows are removed.

In [ ]:
# Identify every row that is missing either the clinical examination date
# or the diagnosis needed to construct a longitudinal diagnosis event.
exception_row_mask = (
    dxsum["EXAMDATE_PARSED"].isna()
    | dxsum["DIAGNOSIS_CODE"].isna()
)

exception_rows = dxsum.loc[exception_row_mask].copy()

# Identify the unique participants represented by at least one exceptional row.
affected_rids = (
    exception_rows["RID_CLEAN"]
    .dropna()
    .unique()
    .tolist()
)

print(f"Rows missing EXAMDATE or DIAGNOSIS: {len(exception_rows):,}")
print(f"Unique participants affected: {len(affected_rids):,}")

print(
    "Unique participants with at least one missing EXAMDATE row:",
    missing_examdate_rows["RID_CLEAN"].nunique(dropna=True),
)

print(
    "Unique participants with at least one missing DIAGNOSIS row:",
    missing_diagnosis_rows["RID_CLEAN"].nunique(dropna=True),
)

print(
    "Unique participants appearing in both categories:",
    len(
        set(
            missing_examdate_rows["RID_CLEAN"]
            .dropna()
            .tolist()
        )
        & set(
            missing_diagnosis_rows["RID_CLEAN"]
            .dropna()
            .tolist()
        )
    ),
)

# Extract the complete DXSUM histories of all affected participants.
affected_participant_timelines = dxsum[
    dxsum["RID_CLEAN"].isin(affected_rids)
].copy()

affected_participant_timelines = (
    affected_participant_timelines
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
        ],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

# Build one subject-level audit row for every affected participant.
subject_audit_records = []

for rid, participant_rows in affected_participant_timelines.groupby(
    "RID_CLEAN",
    sort=True,
):
    participant_rows = participant_rows.copy()

    # A valid diagnosis event must contain both a diagnosis and a clinical date.
    valid_events = participant_rows.dropna(
        subset=[
            "EXAMDATE_PARSED",
            "DIAGNOSIS_CODE",
        ]
    ).sort_values(
        by=[
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
        ],
        kind="stable",
    )

    # A formal baseline event must additionally be labelled as VISCODE2 = bl.
    valid_baseline_events = valid_events[
        valid_events["VISCODE2"].eq("bl")
    ].sort_values(
        by=[
            "EXAMDATE_PARSED",
            "PHASE",
        ],
        kind="stable",
    )

    first_valid_event = (
        valid_events.iloc[0]
        if not valid_events.empty
        else None
    )

    last_valid_event = (
        valid_events.iloc[-1]
        if not valid_events.empty
        else None
    )

    formal_baseline_event = (
        valid_baseline_events.iloc[0]
        if not valid_baseline_events.empty
        else None
    )

    if formal_baseline_event is not None:
        baseline_date = formal_baseline_event["EXAMDATE_PARSED"]

        followup_events = valid_events[
            valid_events["EXAMDATE_PARSED"] > baseline_date
        ]

        if not followup_events.empty:
            final_followup_event = followup_events.iloc[-1]
            last_followup_days = (
                final_followup_event["EXAMDATE_PARSED"]
                - baseline_date
            ).days
        else:
            final_followup_event = None
            last_followup_days = pd.NA
    else:
        followup_events = valid_events.iloc[0:0]
        final_followup_event = None
        last_followup_days = pd.NA

    subject_audit_records.append(
        {
            "RID": int(rid),
            "PTID": (
                participant_rows["PTID"]
                .dropna()
                .iloc[0]
                if participant_rows["PTID"].notna().any()
                else pd.NA
            ),
            "total_dxsum_rows": len(participant_rows),
            "rows_missing_examdate": int(
                participant_rows["EXAMDATE_PARSED"].isna().sum()
            ),
            "rows_missing_diagnosis": int(
                participant_rows["DIAGNOSIS_CODE"].isna().sum()
            ),
            "valid_dated_diagnosis_events": len(valid_events),
            "has_any_valid_dated_diagnosis": not valid_events.empty,
            "has_valid_formal_baseline": not valid_baseline_events.empty,
            "number_of_valid_baseline_rows": len(valid_baseline_events),
            "baseline_date": (
                formal_baseline_event["EXAMDATE_PARSED"]
                if formal_baseline_event is not None
                else pd.NaT
            ),
            "baseline_diagnosis_code": (
                formal_baseline_event["DIAGNOSIS_CODE"]
                if formal_baseline_event is not None
                else pd.NA
            ),
            "baseline_diagnosis": (
                formal_baseline_event["DIAGNOSIS_LABEL"]
                if formal_baseline_event is not None
                else pd.NA
            ),
            "first_valid_diagnosis_date": (
                first_valid_event["EXAMDATE_PARSED"]
                if first_valid_event is not None
                else pd.NaT
            ),
            "first_valid_diagnosis": (
                first_valid_event["DIAGNOSIS_LABEL"]
                if first_valid_event is not None
                else pd.NA
            ),
            "last_valid_diagnosis_date": (
                last_valid_event["EXAMDATE_PARSED"]
                if last_valid_event is not None
                else pd.NaT
            ),
            "last_valid_diagnosis": (
                last_valid_event["DIAGNOSIS_LABEL"]
                if last_valid_event is not None
                else pd.NA
            ),
            "valid_followup_events_after_baseline": len(
                followup_events
            ),
            "has_valid_followup_after_baseline": (
                not followup_events.empty
            ),
            "last_followup_days_from_baseline": last_followup_days,
            "last_followup_months_approx": (
                round(last_followup_days / 30.4375, 1)
                if pd.notna(last_followup_days)
                else pd.NA
            ),
        }
    )

affected_subject_audit = pd.DataFrame(
    subject_audit_records
).sort_values(
    by=[
        "has_valid_formal_baseline",
        "has_valid_followup_after_baseline",
        "RID",
    ],
    ascending=[
        True,
        True,
        True,
    ],
).reset_index(drop=True)

# Add a preliminary interpretation without making a final cohort decision.
def assign_preliminary_status(row):
    if not row["has_any_valid_dated_diagnosis"]:
        return "no_usable_diagnosis_events"

    if not row["has_valid_formal_baseline"]:
        return "no_valid_formal_baseline"

    if not row["has_valid_followup_after_baseline"]:
        return "valid_baseline_but_no_dated_followup"

    return "usable_history_after_ignoring_incomplete_rows"


affected_subject_audit["preliminary_status"] = (
    affected_subject_audit.apply(
        assign_preliminary_status,
        axis=1,
    )
)

# Save both the subject-level summary and the full histories for review.
subject_audit_path = (
    QC_DIR / "dxsum_exception_affected_subject_audit.csv"
)

timeline_output_path = (
    QC_DIR / "dxsum_exception_affected_subject_full_timelines.csv"
)

affected_subject_audit.to_csv(
    subject_audit_path,
    index=False,
)

affected_participant_timelines.to_csv(
    timeline_output_path,
    index=False,
)

print("\nAffected-participant audit completed.")
print(f"Subject-level audit saved to:\n{subject_audit_path}")
print(f"\nFull timelines saved to:\n{timeline_output_path}")

print("\nPreliminary participant status:")
display(
    affected_subject_audit["preliminary_status"]
    .value_counts(dropna=False)
    .rename_axis("preliminary_status")
    .reset_index(name="participant_count")
)

print("\nAffected participants:")
display(affected_subject_audit)

## 8.1. Interpretation of incomplete DXSUM records

The 111 incomplete rows belong to 107 participants, but these participants should not all be removed.

- **54 participants** still have a usable diagnosis history. Only their incomplete rows should be ignored.
- **26 participants** have no usable dated diagnosis and cannot enter the diagnosis-defined cohort.
- **18 participants** have no valid formal baseline diagnosis and cannot enter the baseline prediction cohort.
- **9 participants** have a valid baseline but no dated follow-up. Baseline CN and AD participants may remain in the augmented cohort, but baseline MCI participants cannot be labelled as pMCI or sMCI.

The 54 retained participants must still pass the later trajectory rules for follow-up duration, conversion timing, reversion and diagnostic stability. Missing `EXAMDATE` values should not be replaced with `USERDATE`, because `USERDATE` is an administrative date rather than a confirmed clinical assessment date.

# 9. Assign handling decisions to participants with incomplete DXSUM records

assign a clear handling decision to each participant who has at least one DXSUM row with a missing examination date or diagnosis.

An incomplete row will not automatically remove the participant. Participants with other valid dated diagnosis records will continue into the full clinical trajectory analysis. Participants without any usable diagnosis events or without a valid formal baseline will not enter the baseline-defined cohort.

For participants who have a valid baseline but no dated follow-up, the decision depends on their baseline diagnosis. Baseline CN and AD participants can still contribute to the augmented baseline classification cohort, while baseline MCI participants cannot receive a pMCI or sMCI label without subsequent diagnostic evidence.

In [ ]:
# Assign a transparent handling decision to each affected participant.
def assign_exception_handling(row):
    status = row["preliminary_status"]
    baseline_diagnosis = row["baseline_diagnosis"]

    if status == "usable_history_after_ignoring_incomplete_rows":
        return (
            "retain_participant_ignore_incomplete_rows_"
            "and_continue_to_full_trajectory_rules"
        )

    if status == "no_usable_diagnosis_events":
        return "exclude_no_usable_dated_diagnosis"

    if status == "no_valid_formal_baseline":
        return "exclude_from_baseline_cohort_no_valid_bl_diagnosis"

    if status == "valid_baseline_but_no_dated_followup":
        if baseline_diagnosis in ["CN", "AD"]:
            return (
                "retain_for_augmented_baseline_group_"
                "but_no_longitudinal_outcome"
            )

        if baseline_diagnosis == "MCI":
            return (
                "exclude_from_pmci_smci_"
                "insufficient_diagnostic_followup"
            )

        return "manual_review_unknown_baseline_diagnosis"

    return "manual_review_unrecognised_status"


affected_subject_audit["handling_decision"] = (
    affected_subject_audit.apply(
        assign_exception_handling,
        axis=1,
    )
)

# Display the distribution of decisions.
handling_summary = (
    affected_subject_audit
    .groupby(
        [
            "preliminary_status",
            "baseline_diagnosis",
            "handling_decision",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="participant_count")
    .sort_values(
        by=[
            "preliminary_status",
            "baseline_diagnosis",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Save the updated subject-level audit and its summary.
updated_audit_path = (
    QC_DIR / "dxsum_exception_affected_subject_decisions.csv"
)

handling_summary_path = (
    QC_DIR / "dxsum_exception_handling_summary.csv"
)

affected_subject_audit.to_csv(
    updated_audit_path,
    index=False,
)

handling_summary.to_csv(
    handling_summary_path,
    index=False,
)

print("Handling decisions assigned successfully.")
print(f"\nDetailed decisions saved to:\n{updated_audit_path}")
print(f"\nDecision summary saved to:\n{handling_summary_path}")

print("\nHandling summary:")
display(handling_summary)

print("\nParticipant-level decisions:")
display(
    affected_subject_audit[
        [
            "RID",
            "PTID",
            "baseline_date",
            "baseline_diagnosis",
            "valid_dated_diagnosis_events",
            "valid_followup_events_after_baseline",
            "preliminary_status",
            "handling_decision",
        ]
    ]
)

# 10. Summarise the missing fields for the 54 retained participants

count which core diagnosis fields are missing in the incomplete rows belonging to the 54 participants whose remaining diagnosis histories are usable.

The output will show only the column name, the number of affected rows and the number of affected participants. It will not display the individual records.

In [ ]:
# Identify the 54 participants whose remaining diagnosis histories are usable.
usable_history_rids = affected_subject_audit.loc[
    affected_subject_audit["preliminary_status"].eq(
        "usable_history_after_ignoring_incomplete_rows"
    ),
    "RID",
].astype(int)

# Select only their rows that are incomplete for diagnosis-timeline purposes.
usable_history_incomplete_rows = dxsum[
    dxsum["RID_CLEAN"].isin(usable_history_rids)
    & (
        dxsum["EXAMDATE_PARSED"].isna()
        | dxsum["DIAGNOSIS_CODE"].isna()
    )
].copy()

# Summarise the missing core fields.
missing_column_summary = pd.DataFrame(
    {
        "column_name": [
            "EXAMDATE",
            "DIAGNOSIS",
        ],
        "missing_row_count": [
            int(
                usable_history_incomplete_rows[
                    "EXAMDATE_PARSED"
                ].isna().sum()
            ),
            int(
                usable_history_incomplete_rows[
                    "DIAGNOSIS_CODE"
                ].isna().sum()
            ),
        ],
        "affected_participant_count": [
            usable_history_incomplete_rows.loc[
                usable_history_incomplete_rows[
                    "EXAMDATE_PARSED"
                ].isna(),
                "RID_CLEAN",
            ].nunique(),
            usable_history_incomplete_rows.loc[
                usable_history_incomplete_rows[
                    "DIAGNOSIS_CODE"
                ].isna(),
                "RID_CLEAN",
            ].nunique(),
        ],
    }
)

print(f"Retained participants: {len(usable_history_rids):,}")
print(
    "Incomplete rows belonging to these participants:",
    f"{len(usable_history_incomplete_rows):,}",
)

display(missing_column_summary)

# 11. Check what information remains in the 57 incomplete rows

examine all original DXSUM columns in the 57 incomplete rows belonging to the 54 retained participants.

For each column, count:

- how many of the 57 rows contain a recorded value,
- how many contain a meaningful value after treating blank entries and `-4` as unavailable,
- the percentage of rows containing meaningful information.

This will show whether these rows contain useful clinical or administrative information even though they cannot be used as dated diagnosis events.

In [ ]:
# Restrict the analysis to the original columns present in the raw DXSUM file.
original_dxsum_columns = dxsum_raw.columns.tolist()

incomplete_original_fields = usable_history_incomplete_rows[
    original_dxsum_columns
].copy()

def count_meaningful_values(series):
    """
    Count values that are not missing, not blank strings,
    and not the ADNI unavailable/not-applicable code -4.
    """
    non_missing = series.notna()

    if pd.api.types.is_numeric_dtype(series):
        meaningful = non_missing & series.ne(-4)
    else:
        cleaned = series.astype("string").str.strip()
        meaningful = (
            non_missing
            & cleaned.ne("")
            & cleaned.ne("-4")
            & cleaned.ne("-4.0")
        )

    return int(meaningful.sum())


column_information_summary = pd.DataFrame(
    {
        "column_name": original_dxsum_columns,
        "recorded_row_count": [
            int(incomplete_original_fields[column].notna().sum())
            for column in original_dxsum_columns
        ],
        "meaningful_row_count": [
            count_meaningful_values(
                incomplete_original_fields[column]
            )
            for column in original_dxsum_columns
        ],
    }
)

column_information_summary["meaningful_percentage"] = (
    column_information_summary["meaningful_row_count"]
    / len(incomplete_original_fields)
    * 100
).round(1)

# Show only columns containing information in at least one of the 57 rows.
column_information_summary = (
    column_information_summary[
        column_information_summary["meaningful_row_count"] > 0
    ]
    .sort_values(
        by=[
            "meaningful_row_count",
            "column_name",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print(
    f"Incomplete rows examined: "
    f"{len(incomplete_original_fields):,}"
)

print(
    "Columns containing meaningful information:",
    len(column_information_summary),
)

display(column_information_summary)

# 12. Show each retained participant's earliest REGISTRY record

find the earliest dated REGISTRY record for each of the 54 retained participants and display only the participant ID, ADNI phase and first examination date.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging"
)

RAW_DIR = PROJECT_DIR / "raw"

# Locate the REGISTRY file inside the raw-data folders.
registry_candidates = list(
    RAW_DIR.rglob("*REGISTRY*.csv")
)

if len(registry_candidates) == 0:
    raise FileNotFoundError(
        f"No REGISTRY CSV file was found under:\n{RAW_DIR}"
    )

print("REGISTRY candidates found:")

for path in registry_candidates:
    print(path)

# Use the first matching REGISTRY file.
# Check the printed path before continuing if more than one file appears.
REGISTRY_PATH = registry_candidates[0]

registry_raw = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

print("\nREGISTRY loaded successfully.")
print(f"File: {REGISTRY_PATH}")
print(f"Rows: {len(registry_raw):,}")
print(f"Columns: {registry_raw.shape[1]:,}")

In [ ]:
# Prepare REGISTRY dates.
registry = registry_raw.copy()

registry["RID_CLEAN"] = pd.to_numeric(
    registry["RID"],
    errors="coerce",
).astype("Int64")

registry["REGISTRY_DATE"] = pd.to_datetime(
    registry["USERDATE"],
    errors="coerce",
)

registry["REGISTRY_EXAM_DATE"] = pd.to_datetime(
    registry["EXAMDATE"],
    errors="coerce",
)

# Select the earliest REGISTRY entry for each of the 54 retained participants.
earliest_registry_records = (
    registry[
        registry["RID_CLEAN"].isin(usable_history_rids)
        & registry["REGISTRY_DATE"].notna()
    ]
    .sort_values(
        ["RID_CLEAN", "REGISTRY_DATE"],
        kind="stable",
    )
    .drop_duplicates(
        subset="RID_CLEAN",
        keep="first",
    )
    [
        [
            "RID_CLEAN",
            "PTID",
            "PHASE",
            "REGISTRY_DATE",
            "REGISTRY_EXAM_DATE",
        ]
    ]
    .copy()
)

# Match the DXSUM diagnosis recorded for the same participant
# on the same examination date.
dated_dxsum_diagnoses = (
    dxsum.dropna(
        subset=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "DIAGNOSIS_LABEL",
        ]
    )
    [
        [
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "DIAGNOSIS_LABEL",
        ]
    ]
    .drop_duplicates(
        subset=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
        ],
        keep="first",
    )
    .rename(
        columns={
            "EXAMDATE_PARSED": "REGISTRY_EXAM_DATE",
            "DIAGNOSIS_LABEL": "DIAGNOSIS",
        }
    )
)

earliest_registry_records = earliest_registry_records.merge(
    dated_dxsum_diagnoses,
    on=[
        "RID_CLEAN",
        "REGISTRY_EXAM_DATE",
    ],
    how="left",
)

# Identify each participant's formal DXSUM baseline.
formal_baselines = (
    dxsum[
        dxsum["VISCODE2"].eq("bl")
        & dxsum["EXAMDATE_PARSED"].notna()
        & dxsum["DIAGNOSIS_CODE"].notna()
    ]
    .sort_values(
        ["RID_CLEAN", "EXAMDATE_PARSED"],
        kind="stable",
    )
    .drop_duplicates(
        subset="RID_CLEAN",
        keep="first",
    )
    [
        [
            "RID_CLEAN",
            "EXAMDATE_PARSED",
        ]
    ]
    .rename(
        columns={
            "EXAMDATE_PARSED": "BASELINE_DATE",
        }
    )
)

# Find the final valid dated diagnosis for each participant.
last_dxsum_dates = (
    dxsum.dropna(
        subset=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "DIAGNOSIS_CODE",
        ]
    )
    .groupby(
        "RID_CLEAN",
        as_index=False,
    )["EXAMDATE_PARSED"]
    .max()
    .rename(
        columns={
            "EXAMDATE_PARSED": "LAST_DXSUM_DATE",
        }
    )
)

earliest_registry_records = (
    earliest_registry_records
    .merge(
        formal_baselines,
        on="RID_CLEAN",
        how="left",
    )
    .merge(
        last_dxsum_dates,
        on="RID_CLEAN",
        how="left",
    )
)

# Mark whether valid diagnostic follow-up reaches 36 calendar months.
earliest_registry_records["36_MONTH_FOLLOWUP"] = (
    earliest_registry_records["LAST_DXSUM_DATE"]
    >= (
        earliest_registry_records["BASELINE_DATE"]
        + pd.DateOffset(months=36)
    )
).map(
    {
        True: "Yes",
        False: "No",
    }
)

# Display only the requested columns.
registry_dxsum_comparison = (
    earliest_registry_records[
        [
            "PTID",
            "PHASE",
            "REGISTRY_DATE",
            "REGISTRY_EXAM_DATE",
            "DIAGNOSIS",
            "36_MONTH_FOLLOWUP",
        ]
    ]
    .sort_values("PTID")
    .reset_index(drop=True)
)

display(registry_dxsum_comparison)

# 13. Retain eligible participants from the incomplete-record group

retain only participants who have a REGISTRY examination date, a matched DXSUM diagnosis and at least 36 months of diagnostic follow-up. All other participants in this exceptional-record group will be marked for exclusion.

In [ ]:
# Apply the agreed eligibility rule to the 54 participants.
registry_dxsum_comparison["RETAIN"] = (
    registry_dxsum_comparison["REGISTRY_EXAM_DATE"].notna()
    & registry_dxsum_comparison["DIAGNOSIS"].notna()
    & registry_dxsum_comparison["36_MONTH_FOLLOWUP"].eq("Yes")
).map(
    {
        True: "Yes",
        False: "No",
    }
)

# Separate the participants who satisfy all three conditions.
retained_exception_participants = (
    registry_dxsum_comparison[
        registry_dxsum_comparison["RETAIN"].eq("Yes")
    ]
    .copy()
    .reset_index(drop=True)
)

excluded_exception_participants = (
    registry_dxsum_comparison[
        registry_dxsum_comparison["RETAIN"].eq("No")
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Rows in comparison table: {len(registry_dxsum_comparison)}")
print(
    "Unique participants in comparison table:",
    registry_dxsum_comparison["PTID"].nunique(),
)
print(f"Participants retained: {len(retained_exception_participants)}")
print(f"Participants excluded: {len(excluded_exception_participants)}")

display(retained_exception_participants)

So far, among the 107 participants affected by incomplete DXSUM rows:

26: no usable dated diagnosis
18: no valid formal baseline
9: valid baseline but no follow-up
38: from the remaining 54, fail the current requirement of diagnosis + REGISTRY exam date + 36-month follow-up

Total excluded so far: 91 participants

Retained for later inspection: 16 participants

This exclusion count applies only to this exceptional-record group, not to the full DXSUM population.

# 14. Apply the DXSUM exception decisions

apply the decisions made for participants affected by missing examination dates or diagnoses.

Of the 107 affected participants, 16 will be retained because they have an examination date, a diagnosis and at least 36 months of diagnostic follow-up. The remaining 91 participants will be excluded from the baseline-defined cohort.

also remove individual DXSUM rows that cannot function as diagnosis events because they lack an examination date or diagnosis. The original raw DXSUM table will remain unchanged, and the retained and excluded participant lists will be saved for quality control.

In [ ]:
# Identify the 16 participants retained from the exceptional-record group.
retained_exception_ptids = set(
    registry_dxsum_comparison.loc[
        registry_dxsum_comparison["RETAIN"].eq("Yes"),
        "PTID",
    ]
    .dropna()
    .astype(str)
)

# The other 38 participants from the 54-person comparison are excluded.
comparison_excluded_ptids = set(
    registry_dxsum_comparison.loc[
        registry_dxsum_comparison["RETAIN"].eq("No"),
        "PTID",
    ]
    .dropna()
    .astype(str)
)

# Identify the 53 participants who had already failed the earlier audit:
# no usable diagnosis events, no valid formal baseline, or no follow-up.
previously_excluded_ptids = set(
    affected_subject_audit.loc[
        ~affected_subject_audit["preliminary_status"].eq(
            "usable_history_after_ignoring_incomplete_rows"
        ),
        "PTID",
    ]
    .dropna()
    .astype(str)
)

# Combine both exclusion groups.
excluded_exception_ptids = (
    previously_excluded_ptids
    | comparison_excluded_ptids
)

# Confirm that the subject-level decisions match the expected totals.
assert len(retained_exception_ptids) == 16
assert len(previously_excluded_ptids) == 53
assert len(comparison_excluded_ptids) == 38
assert len(excluded_exception_ptids) == 91
assert retained_exception_ptids.isdisjoint(excluded_exception_ptids)

# Create a subject-level decision table for all 107 affected participants.
exception_subject_decisions = (
    affected_subject_audit[
        [
            "RID",
            "PTID",
            "preliminary_status",
        ]
    ]
    .copy()
)

exception_subject_decisions["exception_decision"] = (
    exception_subject_decisions["PTID"]
    .astype(str)
    .isin(retained_exception_ptids)
    .map(
        {
            True: "retain",
            False: "exclude",
        }
    )
)

# Create the clean working diagnosis-event table.
# A usable event requires a valid participant, date and diagnosis.
valid_event_mask = (
    dxsum["RID_CLEAN"].notna()
    & dxsum["EXAMDATE_PARSED"].notna()
    & dxsum["DIAGNOSIS_CODE"].isin([1, 2, 3])
    & ~dxsum["PTID"].astype(str).isin(excluded_exception_ptids)
)

dxsum_clean_events = (
    dxsum.loc[valid_event_mask]
    .copy()
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Save the participant decisions and clean working table.
decision_output_path = (
    QC_DIR / "dxsum_exception_subject_final_decisions.csv"
)

clean_events_output_path = (
    INTERIM_DIR / "dxsum_clean_diagnosis_events.csv"
)

exception_subject_decisions.to_csv(
    decision_output_path,
    index=False,
)

dxsum_clean_events.to_csv(
    clean_events_output_path,
    index=False,
)

print(f"Affected participants retained: {len(retained_exception_ptids)}")
print(f"Affected participants excluded: {len(excluded_exception_ptids)}")

print(f"\nClean DXSUM diagnosis rows: {len(dxsum_clean_events):,}")
print(
    "Participants remaining in the clean DXSUM table:",
    f"{dxsum_clean_events['RID_CLEAN'].nunique():,}",
)

print(f"\nDecisions saved to:\n{decision_output_path}")
print(f"\nClean diagnosis events saved to:\n{clean_events_output_path}")

# 15. Identify participants with a valid baseline and 36-month diagnosis follow-up

create one record per participant using their earliest valid formal baseline diagnosis from DXSUM.

For each participant, calculate the exact date 36 calendar months after baseline and check whether at least one valid diagnosis is available on or after that date.

This step will show how many CN, MCI and AD participants have sufficient diagnostic follow-up. It will not yet assign pMCI or sMCI labels, because baseline-MCI participants must still be checked for conversion timing, reversion and diagnostic stability.

In [ ]:
# Select valid formal baseline diagnosis records.
formal_baseline_rows = (
    dxsum_clean_events[
        dxsum_clean_events["VISCODE2"].eq("bl")
        & dxsum_clean_events["EXAMDATE_PARSED"].notna()
        & dxsum_clean_events["DIAGNOSIS_CODE"].isin([1, 2, 3])
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
        ],
        kind="stable",
    )
    .copy()
)

# Keep the earliest valid formal baseline record for each participant.
participant_baselines = (
    formal_baseline_rows
    .drop_duplicates(
        subset="RID_CLEAN",
        keep="first",
    )
    [
        [
            "RID_CLEAN",
            "PTID",
            "PHASE",
            "EXAMDATE_PARSED",
            "DIAGNOSIS_CODE",
            "DIAGNOSIS_LABEL",
        ]
    ]
    .rename(
        columns={
            "EXAMDATE_PARSED": "BASELINE_DATE",
            "DIAGNOSIS_CODE": "BASELINE_DIAGNOSIS_CODE",
            "DIAGNOSIS_LABEL": "BASELINE_DIAGNOSIS",
        }
    )
    .reset_index(drop=True)
)

# Define the exact 36-calendar-month follow-up boundary.
participant_baselines["FOLLOWUP_36M_DATE"] = (
    participant_baselines["BASELINE_DATE"]
    + pd.DateOffset(months=36)
)

# Join each participant's valid diagnosis history to their baseline record.
post_baseline_history = dxsum_clean_events.merge(
    participant_baselines[
        [
            "RID_CLEAN",
            "BASELINE_DATE",
            "FOLLOWUP_36M_DATE",
        ]
    ],
    on="RID_CLEAN",
    how="inner",
)

# Keep diagnosis events occurring on or after formal baseline.
post_baseline_history = post_baseline_history[
    post_baseline_history["EXAMDATE_PARSED"]
    >= post_baseline_history["BASELINE_DATE"]
].copy()

# Find the latest valid diagnosis date for each participant.
latest_diagnosis_dates = (
    post_baseline_history
    .groupby(
        "RID_CLEAN",
        as_index=False,
    )["EXAMDATE_PARSED"]
    .max()
    .rename(
        columns={
            "EXAMDATE_PARSED": "LAST_VALID_DIAGNOSIS_DATE",
        }
    )
)

participant_baselines = participant_baselines.merge(
    latest_diagnosis_dates,
    on="RID_CLEAN",
    how="left",
)

# Mark whether the participant has diagnosis follow-up reaching 36 months.
participant_baselines["HAS_36M_DIAGNOSIS_FOLLOWUP"] = (
    participant_baselines["LAST_VALID_DIAGNOSIS_DATE"]
    >= participant_baselines["FOLLOWUP_36M_DATE"]
).map(
    {
        True: "Yes",
        False: "No",
    }
)

# Create a summary by baseline diagnosis and follow-up availability.
baseline_followup_summary = (
    participant_baselines
    .groupby(
        [
            "BASELINE_DIAGNOSIS",
            "HAS_36M_DIAGNOSIS_FOLLOWUP",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="PARTICIPANT_COUNT")
    .sort_values(
        by=[
            "BASELINE_DIAGNOSIS",
            "HAS_36M_DIAGNOSIS_FOLLOWUP",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Save the subject-level table and summary.
participant_baseline_path = (
    INTERIM_DIR / "dxsum_participant_baseline_and_36m_followup.csv"
)

baseline_followup_summary_path = (
    QC_DIR / "dxsum_baseline_36m_followup_summary.csv"
)

participant_baselines.to_csv(
    participant_baseline_path,
    index=False,
)

baseline_followup_summary.to_csv(
    baseline_followup_summary_path,
    index=False,
)

print(
    "Unique participants with a valid formal baseline:",
    f"{len(participant_baselines):,}",
)

print(
    "Participants with diagnosis follow-up reaching 36 months:",
    f"{participant_baselines['HAS_36M_DIAGNOSIS_FOLLOWUP'].eq('Yes').sum():,}",
)

print(
    "Baseline-MCI participants with diagnosis follow-up reaching 36 months:",
    f"{(
        participant_baselines['BASELINE_DIAGNOSIS'].eq('MCI')
        & participant_baselines['HAS_36M_DIAGNOSIS_FOLLOWUP'].eq('Yes')
    ).sum():,}",
)

print("\nBaseline diagnosis and follow-up summary:")
display(baseline_followup_summary)

# 16. Save the baseline and 36-month follow-up clinical manifests

save separate participant manifests for those with valid formal baseline diagnoses and those whose diagnosis histories reach at least 36 months.

also save the baseline-MCI subset with 36-month follow-up as the candidate population for the later pMCI and sMCI trajectory rules.

In [ ]:
# Keep all participants whose diagnosis history reaches 36 months.
participants_with_36m_followup = (
    participant_baselines[
        participant_baselines[
            "HAS_36M_DIAGNOSIS_FOLLOWUP"
        ].eq("Yes")
    ]
    .copy()
    .reset_index(drop=True)
)

# Keep baseline-MCI participants whose diagnosis history reaches 36 months.
baseline_mci_with_36m_followup = (
    participants_with_36m_followup[
        participants_with_36m_followup[
            "BASELINE_DIAGNOSIS"
        ].eq("MCI")
    ]
    .copy()
    .reset_index(drop=True)
)

# Save both manifests.
all_36m_path = (
    MANIFESTS_DIR
    / "clinical_baseline_participants_with_36m_followup.csv"
)

mci_36m_path = (
    MANIFESTS_DIR
    / "baseline_mci_candidates_with_36m_followup.csv"
)

participants_with_36m_followup.to_csv(
    all_36m_path,
    index=False,
)

baseline_mci_with_36m_followup.to_csv(
    mci_36m_path,
    index=False,
)

print(
    "All baseline participants with 36-month follow-up:",
    len(participants_with_36m_followup),
)

print(
    "Baseline-MCI candidates with 36-month follow-up:",
    len(baseline_mci_with_36m_followup),
)

print(f"\nSaved to:\n{all_36m_path}")
print(f"\nSaved to:\n{mci_36m_path}")

# 17. Rebuild the valid DXSUM diagnosis-event table

rebuild the working DXSUM table using row-level validity only.

A diagnosis event is usable when it has:

- a valid participant identifier,
- a DXSUM examination date,
- a valid DXSUM diagnosis code of CN, MCI or AD.

Participants will not be removed simply because they also have an incomplete row elsewhere. The incomplete rows will be preserved separately for quality control.

This corrects the earlier REGISTRY-based filtering. REGISTRY will not determine participant eligibility or diagnosis labels.

In [ ]:
# Define whether each DXSUM row can be used as a dated diagnosis event.
valid_dxsum_event_mask = (
    dxsum["RID_CLEAN"].notna()
    & dxsum["EXAMDATE_PARSED"].notna()
    & dxsum["DIAGNOSIS_CODE"].isin([1, 2, 3])
)

# Keep every usable diagnosis event, regardless of whether the same
# participant also has an incomplete row elsewhere in DXSUM.
dxsum_valid_events = (
    dxsum.loc[valid_dxsum_event_mask]
    .copy()
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Preserve all rows that cannot be used as dated diagnosis events.
dxsum_invalid_event_rows = (
    dxsum.loc[~valid_dxsum_event_mask]
    .copy()
    .reset_index(drop=True)
)

# Record why each invalid row cannot enter the diagnosis timeline.
def describe_invalid_event(row):
    reasons = []

    if pd.isna(row["RID_CLEAN"]):
        reasons.append("missing_rid")

    if pd.isna(row["EXAMDATE_PARSED"]):
        reasons.append("missing_examdate")

    if pd.isna(row["DIAGNOSIS_CODE"]):
        reasons.append("missing_diagnosis")
    elif row["DIAGNOSIS_CODE"] not in [1, 2, 3]:
        reasons.append("unexpected_diagnosis_code")

    return ";".join(reasons)


dxsum_invalid_event_rows["invalid_event_reason"] = (
    dxsum_invalid_event_rows.apply(
        describe_invalid_event,
        axis=1,
    )
)

# Save the corrected working event table and the excluded rows.
valid_events_path = (
    INTERIM_DIR / "dxsum_valid_dated_diagnosis_events.csv"
)

invalid_rows_path = (
    QC_DIR / "dxsum_invalid_diagnosis_event_rows.csv"
)

dxsum_valid_events.to_csv(
    valid_events_path,
    index=False,
)

dxsum_invalid_event_rows.to_csv(
    invalid_rows_path,
    index=False,
)

print(f"Original DXSUM rows: {len(dxsum):,}")
print(f"Usable dated diagnosis rows: {len(dxsum_valid_events):,}")
print(f"Invalid diagnosis-event rows: {len(dxsum_invalid_event_rows):,}")

print(
    "Participants with at least one usable diagnosis event:",
    f"{dxsum_valid_events['RID_CLEAN'].nunique():,}",
)

print(
    "Participants with no usable diagnosis events:",
    f"{dxsum['RID_CLEAN'].nunique() - dxsum_valid_events['RID_CLEAN'].nunique():,}",
)

print(f"\nValid events saved to:\n{valid_events_path}")
print(f"\nInvalid rows saved to:\n{invalid_rows_path}")

# 18. Audit duplicate baseline records and same-day diagnosis conflicts

check whether any participant has more than one valid formal baseline record and whether any participant has conflicting diagnoses recorded on the same calendar date.

These records must be reviewed before selecting one baseline diagnosis per participant and constructing the 36-month outcome labels. No diagnosis will be changed or removed in this step. All identified cases will be saved as quality-control tables.

In [ ]:
# Select all valid formal baseline diagnosis records.
formal_baseline_rows = (
    dxsum_valid_events[
        dxsum_valid_events["VISCODE2"].eq("bl")
    ]
    .copy()
)

# Summarise baseline records at participant level.
baseline_record_summary = (
    formal_baseline_rows
    .groupby(
        "RID_CLEAN",
        as_index=False,
    )
    .agg(
        PTID=("PTID", "first"),
        baseline_row_count=("RID_CLEAN", "size"),
        baseline_date_count=("EXAMDATE_PARSED", "nunique"),
        baseline_diagnosis_count=("DIAGNOSIS_CODE", "nunique"),
        earliest_baseline_date=("EXAMDATE_PARSED", "min"),
        latest_baseline_date=("EXAMDATE_PARSED", "max"),
        phases=(
            "PHASE",
            lambda values: ", ".join(
                sorted(values.dropna().astype(str).unique())
            ),
        ),
        diagnoses=(
            "DIAGNOSIS_LABEL",
            lambda values: ", ".join(
                sorted(values.dropna().astype(str).unique())
            ),
        ),
    )
)

# Keep participants with more than one valid baseline row.
multiple_baseline_subjects = (
    baseline_record_summary[
        baseline_record_summary["baseline_row_count"] > 1
    ]
    .copy()
    .reset_index(drop=True)
)

# Classify the type of baseline ambiguity.
def describe_baseline_issue(row):
    if row["baseline_diagnosis_count"] > 1:
        return "conflicting_baseline_diagnoses"

    if row["baseline_date_count"] > 1:
        return "multiple_baseline_dates_same_diagnosis"

    return "repeated_same_date_same_diagnosis"


multiple_baseline_subjects["baseline_issue"] = (
    multiple_baseline_subjects.apply(
        describe_baseline_issue,
        axis=1,
    )
)

# Extract the original baseline rows for affected participants.
multiple_baseline_rows = (
    formal_baseline_rows[
        formal_baseline_rows["RID_CLEAN"].isin(
            multiple_baseline_subjects["RID_CLEAN"]
        )
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Find participant-date combinations containing more than one diagnosis.
same_day_diagnosis_summary = (
    dxsum_valid_events
    .groupby(
        [
            "RID_CLEAN",
            "EXAMDATE_PARSED",
        ],
        as_index=False,
    )
    .agg(
        diagnosis_count=("DIAGNOSIS_CODE", "nunique"),
        row_count=("DIAGNOSIS_CODE", "size"),
    )
)

same_day_conflict_keys = (
    same_day_diagnosis_summary[
        same_day_diagnosis_summary["diagnosis_count"] > 1
    ]
    [
        [
            "RID_CLEAN",
            "EXAMDATE_PARSED",
        ]
    ]
)

same_day_conflict_rows = (
    dxsum_valid_events.merge(
        same_day_conflict_keys,
        on=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
        ],
        how="inner",
    )
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
            "VISCODE2",
            "DIAGNOSIS_CODE",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Create a concise QC summary.
diagnosis_ambiguity_summary = pd.DataFrame(
    {
        "check": [
            "Participants with at least one valid formal baseline",
            "Participants with multiple valid baseline rows",
            "Participants with conflicting baseline diagnoses",
            "Participants with multiple baseline dates",
            "Participant-date combinations with conflicting diagnoses",
            "Participants involved in same-day diagnosis conflicts",
        ],
        "count": [
            baseline_record_summary["RID_CLEAN"].nunique(),
            len(multiple_baseline_subjects),
            int(
                multiple_baseline_subjects[
                    "baseline_diagnosis_count"
                ].gt(1).sum()
            ),
            int(
                multiple_baseline_subjects[
                    "baseline_date_count"
                ].gt(1).sum()
            ),
            len(same_day_conflict_keys),
            same_day_conflict_rows["RID_CLEAN"].nunique(),
        ],
    }
)

# Save the complete QC tables.
diagnosis_ambiguity_summary.to_csv(
    QC_DIR / "dxsum_baseline_and_same_day_conflict_summary.csv",
    index=False,
)

multiple_baseline_subjects.to_csv(
    QC_DIR / "dxsum_multiple_baseline_subjects.csv",
    index=False,
)

multiple_baseline_rows.to_csv(
    QC_DIR / "dxsum_multiple_baseline_rows.csv",
    index=False,
)

same_day_conflict_rows.to_csv(
    QC_DIR / "dxsum_same_day_conflicting_diagnoses.csv",
    index=False,
)

print("Baseline and same-day diagnosis audit completed.\n")

display(diagnosis_ambiguity_summary)

print("\nParticipants with multiple formal baseline records:")
display(multiple_baseline_subjects)

print("\nSame-day conflicting diagnosis records:")
display(
    same_day_conflict_rows[
        [
            "PHASE",
            "PTID",
            "RID_CLEAN",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE_PARSED",
            "DIAGNOSIS_CODE",
            "DIAGNOSIS_LABEL",
        ]
    ]
)

## 18.1. Interpretation of baseline and same-day diagnosis checks

The formal baseline records are clean: **2,944 participants have one unique valid baseline row**, with no duplicated baseline dates or conflicting baseline diagnoses.

Only two participants have conflicting diagnoses recorded on the same later date:

- RID 1200: CN and MCI on the same day.
- RID 4855: MCI and CN on the same day across ADNI2 and ADNI3.

Both participants were CN at formal baseline, so these conflicts do not affect the baseline-MCI prognosis cohort. They can remain in the baseline CN group, but should be flagged and excluded from any longitudinal or intermediate-stage analysis that depends on the exact ordering of later diagnoses.

# 19. Classify baseline-MCI diagnosis trajectories

identify every participant with a valid MCI diagnosis at the formal DXSUM baseline visit and examine their complete dated diagnosis history.

Each participant will be assigned to one of the following groups:

- `pMCI`: conversion from MCI to AD within 36 calendar months, with no later reversion from AD;
- `sMCI`: no AD conversion and a confirmed MCI diagnosis at or beyond 36 months;
- late conversion after 36 months;
- reversion from MCI to CN;
- reversion after an AD diagnosis;
- unstable diagnosis before conversion;
- insufficient diagnostic follow-up.

The exact 36-month boundary will be calculated from the formal baseline date. Participants with ambiguous same-day diagnoses will be flagged for manual review.

In [ ]:
# Recreate one valid formal baseline record per participant.
formal_baselines = (
    dxsum_valid_events[
        dxsum_valid_events["VISCODE2"].eq("bl")
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "EXAMDATE_PARSED",
            "PHASE",
        ],
        kind="stable",
    )
    .drop_duplicates(
        subset="RID_CLEAN",
        keep="first",
    )
    .copy()
)

# Select every participant diagnosed with MCI at formal baseline.
baseline_mci = (
    formal_baselines[
        formal_baselines["DIAGNOSIS_CODE"].eq(2)
    ]
    [
        [
            "RID_CLEAN",
            "PTID",
            "PHASE",
            "EXAMDATE_PARSED",
        ]
    ]
    .rename(
        columns={
            "PHASE": "BASELINE_PHASE",
            "EXAMDATE_PARSED": "BASELINE_DATE",
        }
    )
    .reset_index(drop=True)
)

baseline_mci["OUTCOME_WINDOW_END"] = (
    baseline_mci["BASELINE_DATE"]
    + pd.DateOffset(months=36)
)

# Identify participants with conflicting diagnoses on the same date.
same_day_conflict_rids = set(
    dxsum_valid_events
    .groupby(
        [
            "RID_CLEAN",
            "EXAMDATE_PARSED",
        ]
    )["DIAGNOSIS_CODE"]
    .nunique()
    .loc[lambda values: values > 1]
    .index
    .get_level_values("RID_CLEAN")
    .tolist()
)

trajectory_records = []

for baseline_row in baseline_mci.itertuples(index=False):
    rid = baseline_row.RID_CLEAN
    baseline_date = baseline_row.BASELINE_DATE
    window_end = baseline_row.OUTCOME_WINDOW_END

    # Extract all valid diagnoses from baseline onwards.
    history = (
        dxsum_valid_events[
            dxsum_valid_events["RID_CLEAN"].eq(rid)
            & (
                dxsum_valid_events["EXAMDATE_PARSED"]
                >= baseline_date
            )
        ]
        .sort_values(
            by=[
                "EXAMDATE_PARSED",
                "PHASE",
                "VISCODE2",
            ],
            kind="stable",
        )
        .copy()
    )

    # Follow-up events occur strictly after the formal baseline date.
    followup = history[
        history["EXAMDATE_PARSED"] > baseline_date
    ].copy()

    ad_events = followup[
        followup["DIAGNOSIS_CODE"].eq(3)
    ]

    cn_events = followup[
        followup["DIAGNOSIS_CODE"].eq(1)
    ]

    mci_events_at_or_after_36m = history[
        history["DIAGNOSIS_CODE"].eq(2)
        & (
            history["EXAMDATE_PARSED"]
            >= window_end
        )
    ]

    first_ad_date = (
        ad_events["EXAMDATE_PARSED"].min()
        if not ad_events.empty
        else pd.NaT
    )

    first_cn_date = (
        cn_events["EXAMDATE_PARSED"].min()
        if not cn_events.empty
        else pd.NaT
    )

    first_mci_at_or_after_36m = (
        mci_events_at_or_after_36m[
            "EXAMDATE_PARSED"
        ].min()
        if not mci_events_at_or_after_36m.empty
        else pd.NaT
    )

    last_diagnosis_date = (
        history["EXAMDATE_PARSED"].max()
        if not history.empty
        else pd.NaT
    )

    last_diagnosis = (
        history.iloc[-1]["DIAGNOSIS_LABEL"]
        if not history.empty
        else pd.NA
    )

    ad_within_36m = (
        pd.notna(first_ad_date)
        and first_ad_date <= window_end
    )

    late_ad_conversion = (
        pd.notna(first_ad_date)
        and first_ad_date > window_end
    )

    any_cn_after_baseline = not cn_events.empty

    cn_before_first_ad = (
        pd.notna(first_ad_date)
        and (
            (
                cn_events["EXAMDATE_PARSED"]
                < first_ad_date
            ).any()
        )
    )

    non_ad_after_first_ad = (
        pd.notna(first_ad_date)
        and (
            (
                history["EXAMDATE_PARSED"]
                > first_ad_date
            )
            & ~history["DIAGNOSIS_CODE"].eq(3)
        ).any()
    )

    has_mci_at_or_after_36m = (
        pd.notna(first_mci_at_or_after_36m)
    )

    has_any_diagnosis_at_or_after_36m = (
        history["EXAMDATE_PARSED"]
        .ge(window_end)
        .any()
    )

    has_same_day_conflict = rid in same_day_conflict_rids

    # Apply mutually exclusive trajectory rules.
    if has_same_day_conflict:
        trajectory_label = "manual_review_same_day_conflict"

    elif (
        pd.notna(first_ad_date)
        and non_ad_after_first_ad
    ):
        trajectory_label = "exclude_reverter_after_ad"

    elif ad_within_36m and cn_before_first_ad:
        trajectory_label = (
            "exclude_unstable_before_conversion"
        )

    elif ad_within_36m:
        trajectory_label = "pMCI"

    elif any_cn_after_baseline:
        trajectory_label = "exclude_reverter_to_cn"

    elif late_ad_conversion:
        trajectory_label = "exclude_late_converter"

    elif has_mci_at_or_after_36m:
        trajectory_label = "sMCI"

    else:
        trajectory_label = "exclude_short_followup"

    trajectory_records.append(
        {
            "RID": int(rid),
            "PTID": baseline_row.PTID,
            "BASELINE_PHASE": baseline_row.BASELINE_PHASE,
            "BASELINE_DATE": baseline_date,
            "OUTCOME_WINDOW_END": window_end,
            "FOLLOWUP_EVENT_COUNT": len(followup),
            "LAST_DIAGNOSIS_DATE": last_diagnosis_date,
            "LAST_DIAGNOSIS": last_diagnosis,
            "FIRST_AD_DATE": first_ad_date,
            "FIRST_CN_DATE": first_cn_date,
            "FIRST_MCI_AT_OR_AFTER_36M": (
                first_mci_at_or_after_36m
            ),
            "AD_WITHIN_36M": ad_within_36m,
            "HAS_MCI_AT_OR_AFTER_36M": (
                has_mci_at_or_after_36m
            ),
            "HAS_ANY_DIAGNOSIS_AT_OR_AFTER_36M": (
                has_any_diagnosis_at_or_after_36m
            ),
            "HAS_SAME_DAY_DIAGNOSIS_CONFLICT": (
                has_same_day_conflict
            ),
            "TRAJECTORY_LABEL": trajectory_label,
        }
    )

baseline_mci_trajectories = pd.DataFrame(
    trajectory_records
)

# Assign the binary prognosis target only to confirmed pMCI and sMCI.
baseline_mci_trajectories["PROGNOSIS_TARGET"] = (
    baseline_mci_trajectories["TRAJECTORY_LABEL"]
    .map(
        {
            "sMCI": 0,
            "pMCI": 1,
        }
    )
    .astype("Int64")
)

# Create a concise outcome summary.
trajectory_summary = (
    baseline_mci_trajectories[
        "TRAJECTORY_LABEL"
    ]
    .value_counts()
    .rename_axis("TRAJECTORY_LABEL")
    .reset_index(name="PARTICIPANT_COUNT")
)

# Save the complete trajectory audit and summary.
trajectory_output_path = (
    MANIFESTS_DIR
    / "baseline_mci_36m_trajectory_labels.csv"
)

trajectory_summary_path = (
    QC_DIR
    / "baseline_mci_36m_trajectory_summary.csv"
)

baseline_mci_trajectories.to_csv(
    trajectory_output_path,
    index=False,
)

trajectory_summary.to_csv(
    trajectory_summary_path,
    index=False,
)

print(
    "Baseline-MCI participants evaluated:",
    f"{len(baseline_mci_trajectories):,}",
)

print(
    "Confirmed pMCI and sMCI participants:",
    f"{baseline_mci_trajectories['PROGNOSIS_TARGET'].notna().sum():,}",
)

print("\nTrajectory classification:")
display(trajectory_summary)

print(f"\nTrajectory labels saved to:\n{trajectory_output_path}")
print(f"\nSummary saved to:\n{trajectory_summary_path}")

# 20. Check pMCI and sMCI counts in the MRI manifest

load the previously created MRI download manifest and count how many participants were labelled as pMCI and sMCI. This is only a comparison with the earlier MRI cohort and will not restrict the current non-imaging cohort.

In [ ]:
MRI_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/adni_mri/manifest/"
    "mri_download_manifest_1661_augmented_subjects.csv"
)

if not MRI_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"The MRI manifest was not found at:\n{MRI_MANIFEST_PATH}"
    )

mri_manifest = pd.read_csv(
    MRI_MANIFEST_PATH,
    low_memory=False,
)

# Find the column containing the CN, AD, sMCI and pMCI group labels.
label_column = None

for column in mri_manifest.columns:
    normalised_values = (
        mri_manifest[column]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if {"pmci", "smci"}.intersection(set(normalised_values)):
        label_column = column
        break

if label_column is None:
    raise KeyError(
        "No column containing pMCI or sMCI labels was found.\n"
        f"Available columns:\n{mri_manifest.columns.tolist()}"
    )

# Standardise the group labels for counting.
mri_manifest["MRI_GROUP"] = (
    mri_manifest[label_column]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(
        {
            "pmci": "pMCI",
            "smci": "sMCI",
            "cn": "CN",
            "ad": "AD",
        }
    )
)

mri_mci_counts = (
    mri_manifest[
        mri_manifest["MRI_GROUP"].isin(["pMCI", "sMCI"])
    ]["MRI_GROUP"]
    .value_counts()
    .reindex(["pMCI", "sMCI"], fill_value=0)
    .rename_axis("GROUP")
    .reset_index(name="PARTICIPANT_COUNT")
)

print(f"Manifest rows: {len(mri_manifest):,}")
print(f"Group-label column used: {label_column}\n")

display(mri_mci_counts)

print(
    "Total pMCI + sMCI:",
    int(mri_mci_counts["PARTICIPANT_COUNT"].sum()),
)

# 21. Compare the new clinical labels with the earlier MRI manifest

compare the newly derived pMCI and sMCI labels with the labels stored in the earlier MRI manifest.

This will show:

- how many newly labelled participants also appear in the MRI manifest,
- how often the old and new labels agree,
- which participants have different labels and need review.

The MRI manifest will be used only for comparison. It will not determine eligibility for the non-imaging cohort.

In [ ]:
# Keep only participants who received a confirmed pMCI or sMCI label
# from the current DXSUM trajectory rules.
current_mci_labels = (
    baseline_mci_trajectories[
        baseline_mci_trajectories["TRAJECTORY_LABEL"].isin(
            ["pMCI", "sMCI"]
        )
    ]
    [
        [
            "RID",
            "PTID",
            "BASELINE_DATE",
            "FIRST_AD_DATE",
            "TRAJECTORY_LABEL",
        ]
    ]
    .rename(
        columns={
            "TRAJECTORY_LABEL": "CURRENT_LABEL",
        }
    )
    .copy()
)

# Keep the pMCI and sMCI labels from the earlier MRI manifest.
previous_mri_labels = (
    mri_manifest[
        mri_manifest["MRI_GROUP"].isin(["pMCI", "sMCI"])
    ]
    [
        [
            "PTID",
            "MRI_GROUP",
        ]
    ]
    .rename(
        columns={
            "MRI_GROUP": "MRI_MANIFEST_LABEL",
        }
    )
    .drop_duplicates(
        subset="PTID",
        keep="first",
    )
    .copy()
)

# Compare labels for participants appearing in both sources.
label_comparison = current_mci_labels.merge(
    previous_mri_labels,
    on="PTID",
    how="outer",
    indicator=True,
)

label_comparison["LABEL_AGREEMENT"] = pd.NA

both_sources_mask = label_comparison["_merge"].eq("both")

label_comparison.loc[
    both_sources_mask,
    "LABEL_AGREEMENT",
] = (
    label_comparison.loc[
        both_sources_mask,
        "CURRENT_LABEL",
    ]
    == label_comparison.loc[
        both_sources_mask,
        "MRI_MANIFEST_LABEL",
    ]
).map(
    {
        True: "Yes",
        False: "No",
    }
)

# Summarise participant overlap.
overlap_summary = pd.DataFrame(
    {
        "comparison_group": [
            "Present in both",
            "Current DXSUM labels only",
            "MRI manifest only",
        ],
        "participant_count": [
            int(label_comparison["_merge"].eq("both").sum()),
            int(label_comparison["_merge"].eq("left_only").sum()),
            int(label_comparison["_merge"].eq("right_only").sum()),
        ],
    }
)

# Create a label agreement table for overlapping participants.
agreement_table = pd.crosstab(
    label_comparison.loc[
        both_sources_mask,
        "MRI_MANIFEST_LABEL",
    ],
    label_comparison.loc[
        both_sources_mask,
        "CURRENT_LABEL",
    ],
    margins=True,
)

# Keep mismatched labels for manual review.
label_mismatches = (
    label_comparison[
        label_comparison["LABEL_AGREEMENT"].eq("No")
    ]
    [
        [
            "PTID",
            "MRI_MANIFEST_LABEL",
            "CURRENT_LABEL",
            "BASELINE_DATE",
            "FIRST_AD_DATE",
        ]
    ]
    .sort_values("PTID")
    .reset_index(drop=True)
)

# Save the complete comparison and the mismatches.
comparison_path = (
    QC_DIR / "current_vs_mri_manifest_mci_label_comparison.csv"
)

mismatch_path = (
    QC_DIR / "current_vs_mri_manifest_mci_label_mismatches.csv"
)

label_comparison.to_csv(
    comparison_path,
    index=False,
)

label_mismatches.to_csv(
    mismatch_path,
    index=False,
)

print("Participant overlap:")
display(overlap_summary)

print("\nLabel agreement among participants present in both:")
display(agreement_table)

print(
    "\nAgreements:",
    int(label_comparison["LABEL_AGREEMENT"].eq("Yes").sum()),
)

print(
    "Mismatches:",
    int(label_comparison["LABEL_AGREEMENT"].eq("No").sum()),
)

print("\nParticipants with different labels:")
display(label_mismatches)

print(f"\nFull comparison saved to:\n{comparison_path}")
print(f"\nMismatches saved to:\n{mismatch_path}")

# 22. Inspect the PTDEMOG demographics table

inspect the PTDEMOG file independently, without merging it with DXSUM or restricting it to any previously defined cohort.

At this stage, examine:

- the number of rows and unique participants,
- participant and visit identifiers,
- available date fields,
- demographic variables,
- missingness in each column,
- whether the table contains any diagnosis-related fields.

No participants, visits or variables will be removed yet.

In [ ]:
# Identify the PTDEMOG file from the raw-file inventory.
ptdemog_candidates = raw_file_inventory[
    raw_file_inventory["filename"].str.contains(
        "PTDEMOG",
        case=False,
        na=False,
    )
].copy()

if len(ptdemog_candidates) != 1:
    display(
        ptdemog_candidates[
            [
                "relative_path",
                "filename",
                "size_mb",
            ]
        ]
    )

    raise RuntimeError(
        f"Expected exactly one PTDEMOG file, but found "
        f"{len(ptdemog_candidates)} candidates."
    )

PTDEMOG_PATH = (
    RAW_DIR
    / ptdemog_candidates.iloc[0]["relative_path"]
)

# Load the table without modifying the raw file.
ptdemog_raw = pd.read_csv(
    PTDEMOG_PATH,
    low_memory=False,
)

print("PTDEMOG loaded successfully.")
print(f"Source file:\n{PTDEMOG_PATH}\n")

print(f"Rows: {len(ptdemog_raw):,}")
print(f"Columns: {ptdemog_raw.shape[1]:,}")

if "RID" in ptdemog_raw.columns:
    print(
        "Unique participants:",
        f"{ptdemog_raw['RID'].nunique(dropna=True):,}",
    )

# Identify columns likely to contain identifiers, visits, dates,
# diagnoses or demographic measurements.
identifier_columns = [
    column
    for column in ptdemog_raw.columns
    if column.upper() in {
        "RID",
        "PTID",
        "ID",
        "SITEID",
    }
]

visit_columns = [
    column
    for column in ptdemog_raw.columns
    if "VISCODE" in column.upper()
]

date_columns = [
    column
    for column in ptdemog_raw.columns
    if any(
        term in column.upper()
        for term in [
            "DATE",
            "USERDATE",
            "STAMP",
        ]
    )
]

diagnosis_like_columns = [
    column
    for column in ptdemog_raw.columns
    if any(
        term in column.upper()
        for term in [
            "DIAG",
            "DX",
        ]
    )
]

print("\nIdentifier columns:")
print(identifier_columns)

print("\nVisit-code columns:")
print(visit_columns)

print("\nPossible date columns:")
print(date_columns)

print("\nDiagnosis-like columns:")
print(diagnosis_like_columns)

print("\nAll column names:")
print(ptdemog_raw.columns.tolist())

print("\nFirst five rows:")
display(ptdemog_raw.head())

# Summarise missingness without removing any variables.
ptdemog_missingness = pd.DataFrame(
    {
        "column_name": ptdemog_raw.columns,
        "missing_count": [
            int(ptdemog_raw[column].isna().sum())
            for column in ptdemog_raw.columns
        ],
        "missing_percentage": [
            round(
                ptdemog_raw[column].isna().mean() * 100,
                2,
            )
            for column in ptdemog_raw.columns
        ],
        "unique_non_missing_values": [
            ptdemog_raw[column].nunique(dropna=True)
            for column in ptdemog_raw.columns
        ],
    }
)

display(ptdemog_missingness)

# 23. Print each PTDEMOG column with one sample value

display every PTDEMOG column together with one non-missing sample entry so I can quickly understand what kind of information each variable contains.

In [ ]:
# Create one sample value for every PTDEMOG column.
column_sample_rows = []

for column in ptdemog_raw.columns:
    non_missing_values = ptdemog_raw[column].dropna()

    sample_value = (
        non_missing_values.iloc[0]
        if not non_missing_values.empty
        else pd.NA
    )

    column_sample_rows.append(
        {
            "COLUMN_NAME": column,
            "SAMPLE_ENTRY": sample_value,
        }
    )

ptdemog_column_samples = pd.DataFrame(column_sample_rows)

# Show every row without pandas collapsing the middle of the table.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 200,
):
    print(
        ptdemog_column_samples.to_string(
            index=False
        )
    )